In [ ]:
# my version -- some problems
import gurobipy as gp
import random
from gurobipy import GRB

# ========= 1. 基本数据（示例，实际请替换） =========
N = 3                      # 管线数量
J = 100                      # 真实事件点数量
T = 100.0                  # 规划期长度



# 集合
N_set = range(N)           # i = 0,...,N-1
J_all = range(J + 2)       # k = 0,...,J+1 （含虚拟）
J_real = range(1, J + 1)   # 真实事件 1,...,J
J_interval = range(J + 1)  # 区间 k=0,...,J 产生泄漏

# 参数（以下仅示例）
b_GR = 3.0
b_RE = 5.0
C_GR = 10.0
C_RE = 100.0
C_Leak = 10.0
M = 3.5                 # 大 M

random.seed(42) 

# P_init = [3.4] * N        # P_i^Init
P_init = [random.uniform(3.21, 3.5) for _ in range(N)]
P_max  = [3.5] * N       # P_i^max
P_min  = [3.0] * N        # P_i^min

# alpha_init = [0.008] * N     # α_i^Init
alpha_init = [random.uniform(0.005, 0.008) for _ in range(N)]
alpha_new  = [alpha_init[i]/4.2 for i in range(N)]     # α_i^new

V = [random.uniform(1, 3) for _ in range(N)] 
K = [10**2 * 146 * V[i] / (8.314 * 293.15) for i in range(N)]   # coefficient to translate pressure to leakage mass

# # 如果每个区间长度相同：
# Delta_t = [T / (J + 1)] * (J + 1)   # k = 0,...,J

# 允许的维护总次数（示例）
L_total = 4

# ========= 2. 创建模型 =========
m = gp.Model("event_based_maintenance")

# ========= 3. 决策变量 =========
# 3.1 维护决策变量 (binary)
x_GR = m.addVars(N_set, J_all, vtype=GRB.BINARY, name="x_GR")
x_RE = m.addVars(N_set, J_all, vtype=GRB.BINARY, name="x_RE")
z = m.addVars(N_set, J_all, vtype=GRB.BINARY, name="z")   # 辅助变量
delta_t = m.addVars(N_set, J_all, vtype=GRB.CONTINUOUS, lb=0.0, name="delta_t")   # k = 0,...,J

# 3.2 时间变量
t      = m.addVars(N_set, J_all, vtype=GRB.CONTINUOUS, lb=0.0, name="t")
t_end  = m.addVars(N_set, J_all, vtype=GRB.CONTINUOUS, lb=0.0, name="t_end")

# 3.3 状态变量
P      = m.addVars(N_set, J_all, vtype=GRB.CONTINUOUS, name="P")
P_end = m.addVars(N_set, J_all, vtype=GRB.CONTINUOUS, name="P_end")
alpha  = m.addVars(N_set, J_all, vtype=GRB.CONTINUOUS, name="alpha")

# 3.4 泄漏量
Leak   = m.addVars(N_set, J_interval, vtype=GRB.CONTINUOUS, lb=0.0, name="Leak")

# ========= 4. 目标函数 =========
m.setObjective(
    gp.quicksum(C_GR * x_GR[i, k] + C_RE * x_RE[i, k] for i in N_set for k in J_interval)
    + gp.quicksum(C_Leak * Leak[i, k] for i in N_set for k in J_interval), GRB.MINIMIZE)

# ========= 5. 约束 =========

# 5.1 虚拟事件起止时间
for i in N_set:
    m.addConstr(t[i, 0]     == 0.0,    name=f"time_start_{i}")
    m.addConstr(t[i, J + 1] == T,      name=f"time_end_{i}")

    # 虚拟事件不能维护
    m.addConstr(x_GR[i, 0] == 0, name=f"dummy0_GR_{i}")
    m.addConstr(x_RE[i, 0] == 0, name=f"dummy0_RE_{i}")
    m.addConstr(x_GR[i, J + 1] == 0, name=f"dummyJ1_GR_{i}")
    m.addConstr(x_RE[i, J + 1] == 0, name=f"dummyJ1_RE_{i}")

    # 虚拟事件结束时间等于本身
    m.addConstr(t_end[i, 0]     == t[i, 0],     name=f"tend0_{i}")
    m.addConstr(t_end[i, J + 1] == t[i, J + 1], name=f"tendJ1_{i}")

# 5.2 真实事件的结束时间
for i in N_set:
    for k in J_real:
        m.addConstr(t_end[i, k] == t[i, k] + b_GR * x_GR[i, k] + b_RE * x_RE[i, k], name=f"tend_{i}_{k}")

# 事件结束时的压力
for i in N_set:
    m.addConstr(P_end[i, 0] == P_init[i], name=f"Pend_0_{i}")
    for k in J_real:
        m.addConstr(P_end[i, k] <= P_max[i] + M * (1 - x_GR[i, k] - x_RE[i, k]), name=f"P_end_max_{i}_{k}")
        m.addConstr(P_end[i, k] >= P_max[i] - M * (1 - x_GR[i, k] - x_RE[i, k]), name=f"P_end2_max_{i}_{k}")
        m.addConstr(P_end[i, k] <= P[i, k] - alpha_init[i] * b_GR * x_GR[i, k] - alpha_new[i] * b_RE * x_RE[i, k] + M * (x_GR[i, k] + x_RE[i, k]), name=f"P_end_{i}_{k}")
        m.addConstr(P_end[i, k] >= P[i, k] - alpha_init[i] * b_GR * x_GR[i, k] - alpha_new[i] * b_RE * x_RE[i, k] + M * (x_GR[i, k] + x_RE[i, k]), name=f"P_end2_{i}_{k}")
        # m.addConstr(P_end[i, k] == P[i, k] * (1 - x_GR[i, k] - x_RE[i, k]) + P_max[i] * (x_GR[i, k] + x_RE[i, k]), name=f"P_end_{i}_{k}")
        

# 5.3 时间非递减 & 不重叠
for i in N_set:
    for k in range(J + 1):   # k = 0,...,J
        m.addConstr(t[i, k] <= t[i, k + 1], name=f"mono_{i}_{k}")
        m.addConstr(t[i, k + 1] >= t_end[i, k] - M * (1 - x_GR[i, k] - x_RE[i, k]), name=f"nonoverlap_{i}_{k}")

# 5.4 每个事件最多一个维护动作
for i in N_set:
    for k in J_real:
        m.addConstr(x_GR[i, k] + x_RE[i, k] <= 1, name=f"one_act_{i}_{k}")

# 5.5 总维护次数约束
m.addConstr(gp.quicksum(x_GR[i, k] + x_RE[i, k] for i in N_set for k in J_real) <= L_total, name="total_maint")

# z definition: if already RE, then alpha is alpha_new else alpha_init

# 如果在 k 之前做过 RE，则 z = 1
for i in N_set:
    m.addConstr(z[i, 1] >= x_RE[i, 1])
    # k = 2..J
    for k in range(2, J + 1):
        m.addConstr(z[i, k] >= z[i, k-1])
        m.addConstr(z[i, k] >= x_RE[i, k])
        m.addConstr(z[i, k] <= z[i, k-1] + x_RE[i, k])

# # delta_t definition
# for i in N_set:
#     m.addConstr(delta_t[i,0] == t[i,1], name=f"delta_t_0_{i}")
#     m.addConstr(delta_t[i,J+1] == T - t[i,J], name=f"delta_t_0_{i}")
#     for k in range(1,J):
#         m.addConstr(delta_t[i, k] == t[i, k+1] - t_end[i, k], name=f"delta_t_def_{i}_{k}")



# 5.6 压力演化与范围
for i in N_set:
    # 初始压力
    m.addConstr(P[i, 0] == P_init[i], name=f"P_init_{i}")

    # 第一段压力（k=0 -> k=1，对应 Δt_0）
    m.addConstr(P[i, 1] == P_init[i] - alpha_init[i] * delta_t[i, 1], name=f"P_first_{i}")

    # 后续段（k=1,...,J）
    # for i in N_set:
    #     for k in range(1, J):  # k = 0,...,J-1，对应 P_{i,k+1}
    #         # 1) 维护时： P_{i,k+1} ≈ P_max[i] - alpha[i,k] * delta_t[k]
    #         m.addConstr(P[i, k+1] <= P_max[i] - alpha[i, k] * delta_t[k] + M * (1 - x_GR[i, k] - x_RE[i, k]), name=f"P_maint_up_{i}_{k}")
    #         m.addConstr(P[i, k+1] >= P_max[i] - alpha[i, k] * delta_t[k] - M * (1 - x_GR[i, k] - x_RE[i, k]), name=f"P_maint_lo_{i}_{k}")

    #         # 2) 不维护时： P_{i,k+1} ≈ P[i,k] - alpha[i,k] * delta_t[k]
    #         m.addConstr(P[i, k+1] <= P[i, k] - alpha[i, k] * delta_t[k] + M * (x_GR[i, k] + x_RE[i, k]), name=f"P_nom_up_{i}_{k}")
    #         m.addConstr(P[i, k+1] >= P[i, k] - alpha[i, k] * delta_t[k] - M * (x_GR[i, k] + x_RE[i, k]), name=f"P_nom_lo_{i}_{k}")


    # # 后续段（k=1,...,J）
    # for k in range(1, J):  # k = 0,...,J-1，对应 P_{i,k+1}
    #     m.addConstr(P[i, k+1] <= P_end[i, k] - alpha_init[i] * delta_t[i, k] + M * z[i, k], name=f"P_maint_up_{i}_{k}")
    #     m.addConstr(P[i, k+1] >= P_end[i, k] - alpha_init[i] * delta_t[i, k] - M * z[i, k], name=f"P_maint_lo_{i}_{k}")
    #     m.addConstr(P[i, k+1] <= P_end[i, k] - alpha_new[i] * delta_t[i, k] + M * (1 - z[i, k]), name=f"P_nom_up_{i}_{k}")
    #     m.addConstr(P[i, k+1] >= P_end[i, k] - alpha_new[i] * delta_t[i, k] - M * (1 - z[i, k]), name=f"P_nom_lo_{i}_{k}")

    for k in range(1, J):
        m.addConstr(P[i, k+1] == P_end[i, k] - (alpha_init[i] * (1 - z[i, k]) + alpha_new[i] * z[i, k]) * delta_t[i, k], name=f"P_evol_{i}_{k}")     # k = 1,...,J，对应 P_{i,k}




    # 压力范围
    for k in J_all:
        m.addConstr(P[i, k] >= P_min[i], name=f"P_min_{i}_{k}")
        m.addConstr(P[i, k] <= P_max[i], name=f"P_max_{i}_{k}")


# for i in N_set:
#     for k in range(J+1):
#         # 情况1：z = 0 → α_init
#         m.addConstr(
#             P[i, k+1] <= P_end[i, k] - alpha_init[i]*delta_t[i, k] + M * z[i, k],
#             name=f"P_alphaInit_up_{i}_{k}"
#         )
#         m.addConstr(
#             P[i, k+1] >= P_end[i, k] - alpha_init[i]*delta_t[i, k] - M * z[i, k],
#             name=f"P_alphaInit_lo_{i}_{k}"
#         )

#         # 情况2：z = 1 → α_new
#         m.addConstr(
#             P[i, k+1] <= P_end[i, k] - alpha_new[i]*delta_t[i, k] + M * (1 - z[i, k]),
#             name=f"P_alphaNew_up_{i}_{k}"
#         )
#         m.addConstr(
#             P[i, k+1] >= P_end[i, k] - alpha_new[i]*delta_t[i, k] - M * (1 - z[i, k]),
#             name=f"P_alphaNew_lo_{i}_{k}"
#         )





# # 5.7 退化率切换
# for i in N_set:
#     # 第一段退化率
#     m.addConstr(alpha[i, 1] == alpha_init[i], name=f"alpha_init_{i}")

#     # k = 2,...,J
#     for k in range(2, J + 1):
#         # α_{i,k} = α_new * sum_{s=1}^k x_RE_{i,s}
#         #         + α_init * (1 - sum_{s=1}^k x_RE_{i,s})
#         m.addConstr(alpha[i, k] == alpha_new[i] * gp.quicksum(x_RE[i, s] for s in range(1, k + 1)) + alpha_init[i] * (1 - gp.quicksum(x_RE[i, s] for s in range(1, k + 1))), name=f"alpha_switch_{i}_{k}")

# 5.8 泄漏计算（区间 k=0,...,J）
# for i in N_set:
#     for k in J_interval:
#         m.addConstr(Leak[i, k] == alpha[i, k] * (t[i, k + 1] - t[i, k]) * K, name=f"leak_{i}_{k}")


# 5.8 泄漏计算（区间 k=0,...,J）
for i in N_set:
    # 区间 0: 用 P_init[i] 到 P[i,1]
    m.addConstr(Leak[i, 0] == 0, name=f"leak_{i}_0")
    m.addConstr(Leak[i, 1] == (P_init[i] - P[i, 1]) * K[i], name=f"leak_{i}_1")

    # 区间 1..J: 用 P_max[i] 到 P[i,k+1]
    for k in range(1, J + 1):
        m.addConstr(Leak[i, k] == (P_max[i] - P[i, k + 1]) * K[i], name=f"leak_{i}_{k}")


# ========= 6. 求解 =========
m.Params.MIPGap = 0.001
# m.Params.NonConvex = 2   #  here if there are non-convex quadratic constraints !!!!!
m.optimize()

# ========= 7. 输出结果 =========
if m.status == GRB.OPTIMAL:
    print(f"Optimal objective: {m.objVal:.4f}")

    # 先按你原来的方式检查所有事件
    for i in N_set:
        print(f"\n==== Caisson {i} all event points ====")
        for k in J_all:
            print(
                f"k={k:2d}, t={t[i,k].X:6.2f}, "
                f"x_GR={int(x_GR[i,k].X)}, x_RE={int(x_RE[i,k].X)}, "
                f"P={P[i,k].X:6.3f}, alpha={alpha[i,k].X:6.4f}"
            )

    # 重点：提取真正的任务安排（只看 k = 1..J、x_GR 或 x_RE 为 1 的点）
    for i in N_set:
        print(f"\n==== Caisson {i} maintenance schedule ====")
        # 收集该 caisson 上所有发生的任务
        tasks_i = []
        for k in J_real:  # 只看真实事件点
            if x_GR[i,k].X > 0.5:
                tasks_i.append((t[i,k].X, k, 'GR'))
            if x_RE[i,k].X > 0.5:
                tasks_i.append((t[i,k].X, k, 'RE'))

        # 按时间排序
        tasks_i.sort(key=lambda x: x[0])

        if not tasks_i:
            print("  No maintenance tasks scheduled.")
        else:
            for time_k, k, typ in tasks_i:
                print(f"  Task at event k={k:2d}, time={time_k:6.2f}, type={typ}")




Set parameter MIPGap to value 0.001
Gurobi Optimizer version 10.0.3 build v10.0.3rc0 (win64)

CPU model: Intel(R) Core(TM) i5-10210U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 4252 rows, 3057 columns and 11766 nonzeros
Model fingerprint: 0x2c9067cf
Model has 297 quadratic constraints
Variable types: 2139 continuous, 918 integer (918 binary)
Coefficient statistics:
  Matrix range     [6e-03, 2e+01]
  QMatrix range    [4e-03, 5e-03]
  QLMatrix range   [6e-03, 1e+00]
  Objective range  [1e+01, 1e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+02]
Presolve removed 370 rows and 981 columns
Presolve time: 0.02s

Explored 0 nodes (0 simplex iterations) in 0.06 seconds (0.01 work units)
Thread count was 1 (of 8 available processors)

Solution count 0

Model is infeasible
Best objective -, best bound -, gap -


In [ ]:
#    test version - it worked
import gurobipy as gp
import random
from gurobipy import GRB

# ========= 1. 基本数据 =========
N = 3                  # 管线数量
J = 20                # 真实事件点数量
T = 100.0              # 规划期长度

# 集合
N_set = range(N)
J_all = range(J + 2)       # k = 0,...,J+1 （含虚拟）
J_real = range(1, J + 1)   # 真实事件 1,...,J
J_interval = range(J + 1)  # 区间 k=0,...,J

# 参数
b_GR = 3.0
b_RE = 5.0
C_GR = 10.0
C_RE = 100.0
C_Leak = 10.0
M = 3.5                 # 大M

random.seed(42)
P_init = [random.uniform(3.21, 3.5) for _ in range(N)]
P_max = [3.5] * N
P_min = [3.0] * N
alpha_init = [random.uniform(0.005, 0.008) for _ in range(N)]
alpha_new = [alpha_init[i]/3 for i in range(N)]
V = [random.uniform(1, 3) for _ in range(N)]
K = [10**2 * 146 * V[i] / (8.314 * 293.15) for i in range(N)]
L_total = 4

# ========= 2. 创建模型 =========
m = gp.Model("event_based_maintenance")
m.Params.MIPGap = 0.001

# ========= 3. 决策变量 =========
x_GR = m.addVars(N_set, J_all, vtype=GRB.BINARY, name="x_GR")
x_RE = m.addVars(N_set, J_all, vtype=GRB.BINARY, name="x_RE")
z = m.addVars(N_set, J_all, vtype=GRB.BINARY, name="z")  # 是否已RE
delta_t = m.addVars(N_set, J_interval, vtype=GRB.CONTINUOUS, lb=0.0, name="delta_t")
t = m.addVars(N_set, J_all, vtype=GRB.INTEGER, lb=0.0, name="t")
t_end = m.addVars(N_set, J_all, vtype=GRB.INTEGER, lb=0.0, name="t_end")
P = m.addVars(N_set, J_all, vtype=GRB.CONTINUOUS, name="P")
P_end = m.addVars(N_set, J_all, vtype=GRB.CONTINUOUS, name="P_end")
Leak = m.addVars(N_set, J_interval, vtype=GRB.CONTINUOUS, lb=0.0, name="Leak")

# ========= 4. 目标函数 =========
m.setObjective(
    gp.quicksum(C_GR * x_GR[i, k] + C_RE * x_RE[i, k] for i in N_set for k in J_real) +
    gp.quicksum(C_Leak * Leak[i, k] for i in N_set for k in J_interval), 
    GRB.MINIMIZE
)

# ========= 5. 约束 =========

# 5.1 虚拟事件约束
for i in N_set:
    m.addConstr(t[i, 0] == 0.0)
    m.addConstr(t[i, J + 1] == T)
    m.addConstr(x_GR[i, 0] == 0)
    m.addConstr(x_RE[i, 0] == 0)
    m.addConstr(x_GR[i, J + 1] == 0)
    m.addConstr(x_RE[i, J + 1] == 0)
    m.addConstr(t_end[i, 0] == t[i, 0])
    m.addConstr(t_end[i, J + 1] == t[i, J + 1])

# 5.2 时间约束：严格递增 + 覆盖整个规划期
# epsilon = 0.005 * T  # 每个区间至少0.5%
epsilon = 1  # 每个区间至少1个时间单位
for i in N_set:
    m.addConstr(t[i, J] >= 0.95 * T, name=f"time_coverage_{i}")  # 确保前J个点覆盖95%
    for k in range(J + 1):
        m.addConstr(t[i, k + 1] >= t[i, k] + epsilon, name=f"time_min_{i}_{k}")
    
    # delta_t 定义
    m.addConstr(delta_t[i, 0] == t[i, 1] - t[i, 0])
    for k in range(1, J):
        m.addConstr(delta_t[i, k] == t[i, k + 1] - t[i, k])
    m.addConstr(delta_t[i, J] == t[i, J + 1] - t[i, J])
    
    # 不重叠约束
    for k in range(J):
        m.addConstr(t[i, k + 1] >= t_end[i, k])

# 5.3 维护结束时间
for i in N_set:
    for k in J_real:
        m.addConstr(t_end[i, k] == t[i, k] + b_GR * x_GR[i, k] + b_RE * x_RE[i, k])

# 5.4 每个事件最多一个维护
for i in N_set:
    for k in J_real:
        m.addConstr(x_GR[i, k] + x_RE[i, k] <= 1)

# 5.5 总维护次数
m.addConstr(gp.quicksum(x_GR[i, k] + x_RE[i, k] for i in N_set for k in J_real) <= L_total)

# 5.6 z变量：累计RE状态
for i in N_set:
    m.addConstr(z[i, 1] >= x_RE[i, 1])
    for k in range(2, J + 1):
        m.addConstr(z[i, k] >= z[i, k - 1])
        m.addConstr(z[i, k] >= x_RE[i, k])
        m.addConstr(z[i, k] <= z[i, k - 1] + x_RE[i, k])

# 5.7 P_end逻辑：维护时=P_max，不维护时=P当前
for i in N_set:
    m.addConstr(P_end[i, 0] == P_init[i])
    for k in J_real:
        m.addConstr(P_end[i, k] == P_max[i] * (x_GR[i, k] + x_RE[i, k]) + 
                           P[i, k] * (1 - x_GR[i, k] - x_RE[i, k]))

# 5.8 压力演化
for i in N_set:
    m.addConstr(P[i, 0] == P_init[i])
    m.addConstr(P[i, 1] == P_init[i] - alpha_init[i] * delta_t[i, 0])
    for k in range(1, J):
        m.addConstr(P[i, k + 1] == P_end[i, k] - 
                   (alpha_init[i] * (1 - z[i, k]) + alpha_new[i] * z[i, k]) * delta_t[i, k])

# 5.9 压力范围
for i in N_set:
    for k in J_all:
        m.addConstr(P[i, k] >= P_min[i])
        m.addConstr(P[i, k] <= P_max[i])

# 5.10 泄漏计算：每个区间的实际压力下降
for i in N_set:
    m.addConstr(Leak[i, 0] == (P_init[i] - P[i, 1]) * K[i])
    for k in range(1, J + 1):
        m.addConstr(Leak[i, k] == (P[i, k] - P[i, k + 1]) * K[i])

# ========= 6. 求解 =========
m.optimize()

# ========= 7. 输出结果 =========
if m.status == GRB.OPTIMAL:
    print(f"Optimal objective: {m.objVal:.4f}")
    
    for i in N_set:
        print(f"\n==== Caisson {i} maintenance schedule ====")
        tasks_i = []
        for k in J_real:
            if x_GR[i, k].X > 0.5:
                tasks_i.append((t[i, k].X, k, 'GR'))
            if x_RE[i, k].X > 0.5:
                tasks_i.append((t[i, k].X, k, 'RE'))
        
        tasks_i.sort(key=lambda x: x[0])
        if not tasks_i:
            print("  No maintenance tasks scheduled.")
        else:
            for time_k, k, typ in tasks_i:
                print(f"  Task at event k={k:2d}, time={time_k:6.2f}, type={typ}")
        
        # 显示关键压力轨迹
        print(f"  Key pressure points:")
        print(f"    t=0: P={P[i,0].X:.3f}")
        print(f"    t={t[i,J].X:.1f}: P={P[i,J].X:.3f}")
        print(f"    t={T}: P={P[i,J+1].X:.3f}")
else:
    print(f"Status: {m.status}")


Set parameter MIPGap to value 0.001
Gurobi Optimizer version 10.0.3 build v10.0.3rc0 (win64)

CPU model: Intel(R) Core(TM) i5-10210U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 712 rows, 588 columns and 1683 nonzeros
Model fingerprint: 0xbe95dd97
Model has 117 quadratic constraints
Variable types: 258 continuous, 330 integer (198 binary)
Coefficient statistics:
  Matrix range     [6e-03, 2e+01]
  QMatrix range    [4e-03, 1e+00]
  QLMatrix range   [6e-03, 4e+00]
  Objective range  [1e+01, 1e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+02]
Presolve removed 138 rows and 228 columns
Presolve time: 0.04s
Presolved: 994 rows, 480 columns, 2985 nonzeros
Variable types: 240 continuous, 240 integer (177 binary)

Root relaxation: objective 1.665406e+01, 542 iterations, 0.02 seconds (0.01 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work

In [ ]:
#    test version - it worked and with better results display
import gurobipy as gp
import random
from gurobipy import GRB

# ========= 1. 基本数据 =========
N = 10                  # 管线数量
# J = 5                # 真实事件点数量
T = 100.0              # 规划期长度

# 集合
N_set = range(N)
J_all = range(J + 2)       # k = 0,...,J+1 （含虚拟）
J_real = range(1, J + 1)   # 真实事件 1,...,J
J_interval = range(J + 1)  # 区间 k=0,...,J

# 参数
b_GR = 3.0
b_RE = 5.0
C_GR = 10.0
C_RE = 100.0
C_Leak = 10.0
M = 3.5                 # big M

random.seed(42)
P_init = [random.uniform(3.21, 3.5) for _ in range(N)]
P_max = [3.5] * N
P_min = [3.01] * N
# alpha_init = [random.uniform(0.005, 0.008) for _ in range(N)]
alpha_init = [random.uniform(0.01, 0.08) for _ in range(N)]
alpha_new = [alpha_init[i]/3 for i in range(N)]
V = [random.uniform(1, 3) for _ in range(N)]
K = [10**2 * 146 * V[i] / (8.314 * 293.15) for i in range(N)]
L_total = 20


T_crit = max((P_init[i]-P_min[i])/alpha_init[i] for i in range(N))
# J_min = max(12, int(T / 15))
# J_max = min(60, int(T / 2.5))
# J = max(J_min, min(J_max, int(T_crit * 1.3 / 2.5)))

J = round(T/T_crit) + 2

total_GE_workers = 20  # 总GE人员数量
total_RE_workers = 10   # 总RE人员数量
b_GR = 3.0             # GR任务持续时间
b_RE = 5.0             # RE任务持续时间

# ========= 2. 创建模型 =========
m = gp.Model("event_based_maintenance")
m.Params.MIPGap = 0.001

# ========= 3. 决策变量 =========
x_GR = m.addVars(N_set, J_all, vtype=GRB.BINARY, name="x_GR")
x_RE = m.addVars(N_set, J_all, vtype=GRB.BINARY, name="x_RE")
z = m.addVars(N_set, J_all, vtype=GRB.BINARY, name="z")  # 是否已RE
delta_t = m.addVars(N_set, J_interval, vtype=GRB.CONTINUOUS, lb=0.0, name="delta_t")
t = m.addVars(N_set, J_all, vtype=GRB.INTEGER, lb=0.0, name="t")
t_end = m.addVars(N_set, J_all, vtype=GRB.INTEGER, lb=0.0, name="t_end")
P = m.addVars(N_set, J_all, vtype=GRB.CONTINUOUS, name="P")
P_end = m.addVars(N_set, J_all, vtype=GRB.CONTINUOUS, name="P_end")
Leak = m.addVars(N_set, J_interval, vtype=GRB.CONTINUOUS, lb=0.0, name="Leak")
GE_workers = m.addVars(N_set, J_all, vtype=GRB.INTEGER, lb=0, ub=total_GE_workers, name="GE_workers")
RE_workers = m.addVars(N_set, J_all, vtype=GRB.INTEGER, lb=0, ub=total_RE_workers, name="RE_workers")

# ========= 4. 目标函数 =========
m.setObjective(
    gp.quicksum(C_GR * x_GR[i, k] + C_RE * x_RE[i, k] for i in N_set for k in J_real) +
    gp.quicksum(C_Leak * Leak[i, k] for i in N_set for k in J_interval), 
    GRB.MINIMIZE
)

# ========= 5. 约束 =========

# 5.1 虚拟事件约束
for i in N_set:
    m.addConstr(t[i, 0] == 0.0)
    m.addConstr(t[i, J + 1] == T)
    m.addConstr(x_GR[i, 0] == 0)
    m.addConstr(x_RE[i, 0] == 0)
    m.addConstr(x_GR[i, J + 1] == 0)
    m.addConstr(x_RE[i, J + 1] == 0)
    m.addConstr(t_end[i, 0] == t[i, 0])
    m.addConstr(t_end[i, J + 1] == t[i, J + 1])

# 5.2 时间约束：严格递增 + 覆盖整个规划期
# epsilon = 0.005 * T  # 每个区间至少0.5%
epsilon = 0.1  # 每个区间至少0.1个时间单位
for i in N_set:
    m.addConstr(t[i, J] >= 0.80 * T, name=f"time_coverage_{i}")  # 确保前J个点覆盖80%
    for k in range(J + 1):
        m.addConstr(t[i, k + 1] >= t[i, k] + epsilon, name=f"time_min_{i}_{k}")
    
    # delta_t 定义
    m.addConstr(delta_t[i, 0] == t[i, 1] - t[i, 0])
    for k in range(1, J):
        m.addConstr(delta_t[i, k] == t[i, k + 1] - t[i, k])
    m.addConstr(delta_t[i, J] == t[i, J + 1] - t[i, J])
    
    # 不重叠约束
    for k in range(J):
        m.addConstr(t[i, k + 1] >= t_end[i, k])

# 5.3 维护结束时间
for i in N_set:
    for k in J_real:
        m.addConstr(t_end[i, k] == t[i, k] + b_GR * x_GR[i, k] + b_RE * x_RE[i, k])

# 5.4 每个事件最多一个维护
for i in N_set:
    for k in J_real:
        m.addConstr(x_GR[i, k] + x_RE[i, k] <= 1)

# 5.5 总维护次数
m.addConstr(gp.quicksum(x_GR[i, k] + x_RE[i, k] for i in N_set for k in J_real) <= L_total)

# 5.6 z变量：累计RE状态
for i in N_set:
    m.addConstr(z[i, 1] >= x_RE[i, 1])
    for k in range(2, J + 1):
        m.addConstr(z[i, k] >= z[i, k - 1])
        m.addConstr(z[i, k] >= x_RE[i, k])
        m.addConstr(z[i, k] <= z[i, k - 1] + x_RE[i, k])

# 5.7 P_end逻辑：维护时=P_max，不维护时=P当前
for i in N_set:
    m.addConstr(P_end[i, 0] == P_init[i])
    for k in J_real:
        m.addConstr(P_end[i, k] == P_max[i] * (x_GR[i, k] + x_RE[i, k]) + 
                           P[i, k] * (1 - x_GR[i, k] - x_RE[i, k]))

# 5.8 压力演化
for i in N_set:
    m.addConstr(P[i, 0] == P_init[i])
    m.addConstr(P[i, 1] == P_init[i] - alpha_init[i] * delta_t[i, 0])
    for k in range(1, J):
        m.addConstr(P[i, k + 1] == P_end[i, k] - (alpha_init[i] * (1 - z[i, k]) + alpha_new[i] * z[i, k]) * delta_t[i, k])

# 5.9 压力范围
for i in N_set:
    for k in J_all:
        m.addConstr(P[i, k] >= P_min[i])
        m.addConstr(P[i, k] <= P_max[i])

# 5.10 泄漏计算：每个区间的实际压力下降
for i in N_set:
    m.addConstr(Leak[i, 0] == (P_init[i] - P[i, 1]) * K[i])
    for k in range(1, J + 1):
        m.addConstr(Leak[i, k] == (P[i, k] - P[i, k + 1]) * K[i])

### 人员约束 ###
# # 5.11 人员需求约束：只有执行任务时才分配人员
# for i in N_set:
#     for k in J_real:
#         # GR任务需要3人，RE任务需要5人
#         m.addConstr(GE_workers[i, k] >= 3 * x_GR[i, k])
#         m.addConstr(GE_workers[i, k] <= 3 * x_GR[i, k] * total_GE_workers)
#         m.addConstr(RE_workers[i, k] >= 5 * x_RE[i, k])
#         m.addConstr(RE_workers[i, k] <= 5 * x_RE[i, k] * total_RE_workers)
        
#         # 无任务时人员为0
#         m.addConstr(GE_workers[i, k] <= total_GE_workers * (x_GR[i, k] + x_RE[i, k]))
#         m.addConstr(RE_workers[i, k] <= total_RE_workers * (x_GR[i, k] + x_RE[i, k]))

# 5.12 防止同一时刻过多任务开始（跨管线）
for k in J_real:
    m.addConstr(gp.quicksum(x_GR[i, k] for i in N_set) <= 8)  # 最多3个GR
    m.addConstr(gp.quicksum(x_RE[i, k] for i in N_set) <= 5)  # 最多1个RE

# # 5.13'''' 每个任务开始时，检查**其他管线**的近期任务
# check_window = 8  # 检查前8个事件（覆盖b_max=5）

# for i_ref in N_set:        # 参考管线
#     for k_ref in J_real:   # 参考任务
#         start_k = max(1, k_ref - check_window)
#         recent_ks = range(start_k, k_ref + 1)
        
#         # 只计算**其他管线**i ≠ i_ref 的近期任务需求
#         GE_demand_other = gp.quicksum(
#             3 * x_GR[i, k] for i in N_set if i != i_ref 
#             for k in recent_ks
#         )
#         RE_demand_other = gp.quicksum(
#             5 * x_RE[i, k] for i in N_set if i != i_ref 
#             for k in recent_ks
#         )
        
#         # 参考任务开始时，其他管线需求 + 本任务需求 <= 总人数
#         m.addConstr(GE_demand_other + 3 * x_GR[i_ref, k_ref] <= total_GE_workers)
#         m.addConstr(RE_demand_other + 5 * x_RE[i_ref, k_ref] <= total_RE_workers)


# # 5.14 虚拟事件点人员为0
# for i in N_set:
#     m.addConstr(GE_workers[i, 0] == 0)
#     m.addConstr(RE_workers[i, 0] == 0)
#     m.addConstr(GE_workers[i, J+1] == 0)
#     m.addConstr(RE_workers[i, J+1] == 0)

# ========= 6. 求解 =========
m.optimize()

# # ========= 7. 输出结果 =========
# if m.status == GRB.OPTIMAL:
#     print(f"Optimal objective: {m.objVal:.4f}")
#     print(f"Total cost = Maintenance({C_GR+C_RE:.1f}x) + Leakage({C_Leak:.1f}x)")
    
#     for i in N_set:
#         print(f"\n{'='*60}")
#         print(f"CAISSON {i} - P_init={P_init[i]:.3f}, α_init={alpha_init[i]:.4f}")
#         print(f"{'='*60}")
        
#         # 1. 维护任务汇总（按时间排序）
#         print("📋 MAINTENANCE SCHEDULE:")
#         tasks_i = []
#         for k in J_real:
#             if x_GR[i, k].X > 0.5:
#                 tasks_i.append((t[i, k].X, k, 'GR', P[i,k].X, P_end[i,k].X))
#             if x_RE[i, k].X > 0.5:
#                 tasks_i.append((t[i, k].X, k, 'RE', P[i,k].X, P_end[i,k].X))
#         tasks_i.sort(key=lambda x: x[0])
        
#         if not tasks_i:
#             print("  ❌ No maintenance tasks scheduled")
#         else:
#             for time_k, k, typ, P_start, P_end_val in tasks_i:
#                 print(f"  🔧 k={k:3d} | t={time_k:6.1f} | {typ} | "
#                       f"P_start={P_start:5.3f} → P_end={P_end_val:5.3f}")
        
#         print()
        
#         # 2. 每个潜在任务点详细决策（前10个 + 有维护的 + 后5个）
#         print("📊 ALL POTENTIAL TASK POINTS:")
#         show_points = list(range(1, 11)) + list(range(J-4, J+1))  # 前10+后5
        
#         # 添加所有有维护的点
#         for _, k, _, _, _ in tasks_i:
#             if k not in show_points:
#                 show_points.append(k)
#         show_points = sorted(set(show_points))
        
#         for k in show_points:
#             if k in J_real:
#                 maint_type = 'GR' if x_GR[i,k].X > 0.5 else 'RE' if x_RE[i,k].X > 0.5 else 'NONE'
#                 deltaP = P_end[i,k].X - P[i,k].X
#                 print(f"  k={k:3d} | t={t[i,k].X:6.1f} | "
#                       f"δt={delta_t[i,k-1].X:5.2f} | "
#                       f"P={P[i,k].X:5.3f} | "
#                       f"maint={maint_type} | "
#                       f"P_end={P_end[i,k].X:5.3f} | "
#                       f"ΔP={deltaP:+6.3f}")
        
#         # 3. 压力轨迹摘要
#         print()
#         print("📈 PRESSURE TRAJECTORY SUMMARY:")
#         print(f"  t=0.0:   P={P[i,0].X:5.3f}")
#         print(f"  t={t[i,J//4].X:5.1f}: P={P[i,J//4].X:5.3f}")
#         print(f"  t={t[i,J//2].X:5.1f}: P={P[i,J//2].X:5.3f}")
#         print(f"  t={t[i,3*J//4].X:5.1f}: P={P[i,3*J//4].X:5.3f}")
#         print(f"  t={t[i,J].X:5.1f}: P={P[i,J].X:5.3f}")
#         print(f"  t={T:5.1f}: P={P[i,J+1].X:5.3f}")
        
#         # 4. 总泄漏统计
#         total_leak = sum(Leak[i,k].X for k in J_interval)
#         print(f"\n💧 TOTAL LEAKAGE: {total_leak:.3f}")

        
# else:
#     print(f"❌ Optimization failed. Status: {m.status}")
#     print("💡 Check constraints or increase MIPGap/time limit")


# ========= 7. 输出结果（修正Workers显示）=========
if m.status == GRB.OPTIMAL:
    print(f"Optimal objective: {m.objVal:.4f}")
    print(f"Total cost = Maintenance({C_GR+C_RE:.1f}x) + Leakage({C_Leak:.1f}x)")
    
    # 👷 全局人员统计
    print(f"\n👷 PERSONNEL SUMMARY:")
    print(f"  GE workers available: {total_GE_workers}")
    print(f"  RE workers available: {total_RE_workers}")
    print()
    
    for i in N_set:
        print(f"\n{'='*70}")
        print(f"CAISSON {i} - P_init={P_init[i]:.3f}, α_init={alpha_init[i]:.4f}")
        print(f"{'='*70}")
        
        # 1. 维护任务汇总（正确显示工人数量）
        print("📋 MAINTENANCE SCHEDULE:")
        tasks_i = []
        for k in J_real:
            if x_GR[i, k].X > 0.5:
                tasks_i.append((t[i, k].X, k, 'GR', P[i,k].X, P_end[i,k].X, 3))  # GR固定3人
            if x_RE[i, k].X > 0.5:
                tasks_i.append((t[i, k].X, k, 'RE', P[i,k].X, P_end[i,k].X, 5))  # RE固定5人
        tasks_i.sort(key=lambda x: x[0])
        
        if not tasks_i:
            print("  ❌ No maintenance tasks scheduled")
        else:
            print(f"  {'k':>3} | {'t':>7} | {'Type':>3} | {'Workers':>8} | {'P_start':>8} → {'P_end':>8}")
            print(f"  {'-'*60}")
            for time_k, k, typ, P_start, P_end_val, workers in tasks_i:
                print(f"  {k:3d} | {time_k:7.1f} | {typ:>3} | {workers:6d} | "
                      f"{P_start:7.3f} → {P_end_val:7.3f}")
        
        print()
        
        # 2. 每个潜在任务点详细决策（显示计划工人）
        print("📊 ALL POTENTIAL TASK POINTS:")
        show_points = list(range(1, 11)) + list(range(J-4, J+1))
        for _, k, _, _, _, _ in tasks_i:
            if k not in show_points:
                show_points.append(k)
        show_points = sorted(set(show_points))
        
        print(f"  {'k':>3} | {'t':>7} | {'δt':>5} | {'P':>6} | {'Type':>6} | {'Workers':>8} | {'P_end':>8} | {'ΔP':>7}")
        print(f"  {'-'*80}")
        for k in show_points:
            if k in J_real:
                if x_GR[i,k].X > 0.5:
                    maint_type, workers = 'GR', 3
                elif x_RE[i,k].X > 0.5:
                    maint_type, workers = 'RE', 5
                else:
                    maint_type, workers = 'NONE', 0
                
                deltaP = P_end[i,k].X - P[i,k].X
                print(f"  {k:3d} | {t[i,k].X:7.1f} | {delta_t[i,k-1].X:5.2f} | "
                      f"{P[i,k].X:6.3f} | {maint_type:>6} | {workers:6d} | "
                      f"{P_end[i,k].X:8.3f} | {deltaP:+7.3f}")
        
        # 3. 压力轨迹摘要
        print()
        print("📈 PRESSURE TRAJECTORY SUMMARY:")
        print(f"  t=0.0:   P={P[i,0].X:5.3f}")
        print(f"  t={t[i,J//4].X:5.1f}: P={P[i,J//4].X:5.3f}")
        print(f"  t={t[i,J//2].X:5.1f}: P={P[i,J//2].X:5.3f}")
        print(f"  t={t[i,3*J//4].X:5.1f}: P={P[i,3*J//4].X:5.3f}")
        print(f"  t={t[i,J].X:5.1f}: P={P[i,J].X:5.3f}")
        print(f"  t={T:5.1f}: P={P[i,J+1].X:5.3f}")
        
        # 4. 只保留泄漏统计（删除total workers）
        total_leak = sum(Leak[i,k].X for k in J_interval)
        print(f"\n💧 LEAKAGE: {total_leak:.3f}")
        
else:
    print(f"❌ Optimization failed. Status: {m.status}")
    print("💡 Check constraints or increase MIPGap/time limit")





Set parameter MIPGap to value 0.001
Gurobi Optimizer version 10.0.3 build v10.0.3rc0 (win64)

CPU model: Intel(R) Core(TM) i5-10210U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 843 rows, 860 columns and 1950 nonzeros
Model fingerprint: 0x8801fba2
Model has 110 quadratic constraints
Variable types: 300 continuous, 560 integer (240 binary)
Coefficient statistics:
  Matrix range     [1e-02, 2e+01]
  QMatrix range    [7e-03, 1e+00]
  QLMatrix range   [1e-02, 4e+00]
  Objective range  [1e+01, 1e+02]
  Bounds range     [1e+00, 2e+01]
  RHS range        [1e-01, 1e+02]
Presolve removed 330 rows and 510 columns
Presolve time: 0.02s
Presolved: 933 rows, 470 columns, 2880 nonzeros
Variable types: 240 continuous, 230 integer (170 binary)

Root relaxation: objective 3.646083e+02, 428 iterations, 0.01 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work